# Semana 3: EDA Avanzado, Estacionariedad y Pregunta de Investigación

## Proyecto: Serie de Tiempo de Caudal – Estación Pueblo Nuevo (IDEAM)

**Objetivo:** Realizar un análisis exploratorio profundo de la serie limpia para identificar patrones temporales, evaluar estacionariedad y formular la pregunta de investigación que guiará el modelado predictivo.

### Contenido:
1. Importar librerías
2. Cargar serie limpia (exportada en Semana 2)
3. Descomposición estacional (STL: Trend + Seasonal + Residual)
4. Visualización de componentes descompuestos
5. Autocorrelación (ACF) y Autocorrelación Parcial (PACF)
6. Pruebas de estacionariedad (ADF y KPSS)
7. Pregunta de investigación
8. Análisis de estacionalidad mensual y trimestral
9. Patrones interanuales y tendencia
10. Distribución por épocas hidrológicas (seca vs lluviosa)
11. Resumen del EDA y hallazgos clave
12. Conclusiones y próximos pasos

## 1. Importar Librerías

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Análisis de series de tiempo
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import warnings
warnings.filterwarnings("ignore")

import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías importadas")

✅ Librerías importadas


## 2. Cargar Serie Limpia (Exportada en Semana 2)

Usamos el CSV `caudal_limpio_diario.csv` generado en la Semana 2, que contiene la serie diaria completa con imputación lineal y capping aplicados.

In [2]:
# Cargar serie limpia
df = pd.read_csv("../Week_2/caudal_limpio_diario.csv", index_col="Fecha", parse_dates=True)

print(f"✅ Serie limpia cargada")
print(f"   Período:  {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros: {len(df)}")
print(f"   Columnas:  {list(df.columns)}")
print(f"   NaN:        {df['Caudal'].isna().sum()}")
print(f"   Frecuencia: {df.index.inferred_freq}")
print(f"\n📊 Estadísticas rápidas:")
df["Caudal"].describe()

✅ Serie limpia cargada
   Período:  2010-01-01 → 2017-12-31
   Registros: 2922
   Columnas:  ['Caudal', 'Caudal_log']
   NaN:        0
   Frecuencia: D

📊 Estadísticas rápidas:


count    2922.000000
mean        3.466200
std         1.662589
min         0.149659
25%         2.633083
50%         3.336000
75%         4.058508
max         9.388000
Name: Caudal, dtype: float64

## 3. Descomposición Estacional (STL)

La descomposición **STL** (*Seasonal and Trend decomposition using Loess*) separa la serie en tres componentes:

- **Tendencia (Trend):** Movimiento de largo plazo de la serie
- **Estacionalidad (Seasonal):** Patrón cíclico que se repite cada año (período = 365 días)
- **Residuo (Residual):** Variación no explicada por tendencia ni estacionalidad

$$Y_t = T_t + S_t + R_t$$

In [3]:
# Descomposición STL (período = 365 días para estacionalidad anual)
stl = STL(df["Caudal"], period=365, robust=True)
resultado = stl.fit()

print("✅ Descomposición STL completada")
print(f"   Período estacional: 365 días")
print(f"   Fuerza de tendencia:      {1 - resultado.resid.var() / (resultado.trend + resultado.resid).var():.3f}")
print(f"   Fuerza de estacionalidad: {1 - resultado.resid.var() / (resultado.seasonal + resultado.resid).var():.3f}")

✅ Descomposición STL completada
   Período estacional: 365 días
   Fuerza de tendencia:      0.290
   Fuerza de estacionalidad: -0.032


## 4. Visualización de Componentes Descompuestos

Cada componente se grafica por separado para entender su contribución a la serie original.

In [4]:
# Visualización de los 4 componentes STL
componentes = {
    "Serie Original": df["Caudal"],
    "Tendencia": resultado.trend,
    "Estacionalidad": resultado.seasonal,
    "Residuo": resultado.resid,
}

colores = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=list(componentes.keys()),
                    vertical_spacing=0.05)

for i, (nombre, serie) in enumerate(componentes.items(), 1):
    fig.add_trace(go.Scatter(
        x=serie.index, y=serie.values,
        mode="lines", name=nombre,
        line=dict(color=colores[i-1], width=0.8),
    ), row=i, col=1)

fig.update_layout(
    title="Descomposición STL de la Serie de Caudal (período = 365 días)",
    height=800, width=1000, showlegend=False,
)
fig.update_yaxes(title_text="m³/s", row=1, col=1)
fig.update_yaxes(title_text="m³/s", row=2, col=1)
fig.update_yaxes(title_text="m³/s", row=3, col=1)
fig.update_yaxes(title_text="m³/s", row=4, col=1)
fig.show()

## 5. Autocorrelación (ACF) y Autocorrelación Parcial (PACF)

- **ACF** (*Autocorrelation Function*): mide la correlación entre $Y_t$ y $Y_{t-k}$ para distintos lags $k$. Picos periódicos indican estacionalidad.
- **PACF** (*Partial ACF*): mide la correlación directa entre $Y_t$ y $Y_{t-k}$ eliminando efectos intermedios. Ayuda a determinar el orden $p$ de modelos AR.

Analizamos hasta **730 lags** (2 años) para captar el ciclo anual completo.

In [5]:
# Calcular ACF y PACF
n_lags = 730
acf_vals = acf(df["Caudal"], nlags=n_lags, fft=True)
pacf_vals = pacf(df["Caudal"], nlags=60)  # PACF solo para lags cortos

# Intervalo de confianza 95%
ic_95 = 1.96 / np.sqrt(len(df))

# ACF
fig_acf = go.Figure()
fig_acf.add_trace(go.Bar(
    x=list(range(n_lags + 1)), y=acf_vals,
    marker_color=["#d62728" if abs(v) > ic_95 else "#aec7e8" for v in acf_vals],
    name="ACF",
))
fig_acf.add_hline(y=ic_95, line_dash="dash", line_color="red", annotation_text="IC 95%")
fig_acf.add_hline(y=-ic_95, line_dash="dash", line_color="red")
fig_acf.add_hline(y=0, line_color="black", line_width=0.5)

# Marcar lags anuales
for yr in [365, 730]:
    fig_acf.add_vline(x=yr, line_dash="dot", line_color="green",
                      annotation_text=f"Lag {yr}")

fig_acf.update_layout(
    title=f"Función de Autocorrelación (ACF) – {n_lags} lags",
    xaxis_title="Lag (días)", yaxis_title="Autocorrelación",
    width=1000, height=400,
)
fig_acf.show()

# PACF (primeros 60 lags)
fig_pacf = go.Figure()
fig_pacf.add_trace(go.Bar(
    x=list(range(len(pacf_vals))), y=pacf_vals,
    marker_color=["#d62728" if abs(v) > ic_95 else "#aec7e8" for v in pacf_vals],
    name="PACF",
))
fig_pacf.add_hline(y=ic_95, line_dash="dash", line_color="red", annotation_text="IC 95%")
fig_pacf.add_hline(y=-ic_95, line_dash="dash", line_color="red")
fig_pacf.add_hline(y=0, line_color="black", line_width=0.5)
fig_pacf.update_layout(
    title="Función de Autocorrelación Parcial (PACF) – 60 lags",
    xaxis_title="Lag (días)", yaxis_title="PACF",
    width=1000, height=400,
)
fig_pacf.show()

print(f"📌 ACF en lag 1: {acf_vals[1]:.4f}")
print(f"📌 ACF en lag 365: {acf_vals[365]:.4f}")
print(f"📌 ACF en lag 730: {acf_vals[730]:.4f}")
print(f"📌 PACF en lag 1: {pacf_vals[1]:.4f}")

📌 ACF en lag 1: 0.9343
📌 ACF en lag 365: 0.0395
📌 ACF en lag 730: -0.1034
📌 PACF en lag 1: 0.9346


## 6. Pruebas de Estacionariedad (ADF y KPSS)

Una serie es **estacionaria** si sus propiedades estadísticas (media, varianza) no cambian con el tiempo.  
Esto es requisito para modelos ARIMA/SARIMA.

| Prueba | $H_0$ (Hipótesis nula) | Criterio |
|--------|----------------------|----------|
| **ADF** (Augmented Dickey-Fuller) | La serie tiene raíz unitaria (NO estacionaria) | Rechazar si $p < 0.05$ → estacionaria |
| **KPSS** (Kwiatkowski–Phillips–Schmidt–Shin) | La serie ES estacionaria | Rechazar si $p < 0.05$ → NO estacionaria |

Aplicamos ambas pruebas sobre la serie **original** y la **diferenciada**.

In [6]:
# Función para aplicar pruebas ADF y KPSS
def pruebas_estacionariedad(serie, nombre="Serie"):
    print(f"\n{'='*60}")
    print(f"📊 PRUEBAS DE ESTACIONARIEDAD: {nombre}")
    print(f"{'='*60}")

    # ADF
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(serie.dropna(), autolag="AIC")
    print(f"\n🔹 Prueba ADF (Augmented Dickey-Fuller):")
    print(f"   Estadístico: {adf_stat:.4f}")
    print(f"   p-valor:     {adf_p:.6f}")
    print(f"   Lags usados: {adf_lags}")
    print(f"   Valores críticos:")
    for key, val in adf_crit.items():
        print(f"     {key}: {val:.4f}")
    adf_result = "✅ ESTACIONARIA" if adf_p < 0.05 else "❌ NO estacionaria"
    print(f"   Conclusión ADF: {adf_result}")

    # KPSS
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(serie.dropna(), regression="c", nlags="auto")
    print(f"\n🔹 Prueba KPSS:")
    print(f"   Estadístico: {kpss_stat:.4f}")
    print(f"   p-valor:     {kpss_p:.4f}")
    print(f"   Lags usados: {kpss_lags}")
    print(f"   Valores críticos:")
    for key, val in kpss_crit.items():
        print(f"     {key}: {val:.4f}")
    kpss_result = "✅ ESTACIONARIA" if kpss_p >= 0.05 else "❌ NO estacionaria"
    print(f"   Conclusión KPSS: {kpss_result}")

    return adf_p, kpss_p

# Serie original
adf_p_orig, kpss_p_orig = pruebas_estacionariedad(df["Caudal"], "Caudal Original")

# Serie diferenciada (orden 1)
df["Caudal_diff"] = df["Caudal"].diff()
adf_p_diff, kpss_p_diff = pruebas_estacionariedad(df["Caudal_diff"].dropna(), "Caudal Diferenciado (d=1)")


📊 PRUEBAS DE ESTACIONARIEDAD: Caudal Original

🔹 Prueba ADF (Augmented Dickey-Fuller):
   Estadístico: -4.6512
   p-valor:     0.000104
   Lags usados: 13
   Valores críticos:
     1%: -3.4326
     5%: -2.8625
     10%: -2.5673
   Conclusión ADF: ✅ ESTACIONARIA

🔹 Prueba KPSS:
   Estadístico: 1.2868
   p-valor:     0.0100
   Lags usados: 31
   Valores críticos:
     10%: 0.3470
     5%: 0.4630
     2.5%: 0.5740
     1%: 0.7390
   Conclusión KPSS: ❌ NO estacionaria

📊 PRUEBAS DE ESTACIONARIEDAD: Caudal Diferenciado (d=1)

🔹 Prueba ADF (Augmented Dickey-Fuller):
   Estadístico: -17.2299
   p-valor:     0.000000
   Lags usados: 16
   Valores críticos:
     1%: -3.4326
     5%: -2.8625
     10%: -2.5673
   Conclusión ADF: ✅ ESTACIONARIA

🔹 Prueba KPSS:
   Estadístico: 0.0382
   p-valor:     0.1000
   Lags usados: 47
   Valores críticos:
     10%: 0.3470
     5%: 0.4630
     2.5%: 0.5740
     1%: 0.7390
   Conclusión KPSS: ✅ ESTACIONARIA


In [7]:
# Visualización: media y varianza móvil para verificar estacionariedad visualmente
window = 365

rolling_mean = df["Caudal"].rolling(window=window).mean()
rolling_std = df["Caudal"].rolling(window=window).std()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Media Móvil (ventana = 365 días)",
                                    "Desviación Estándar Móvil (ventana = 365 días)"],
                    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=df.index, y=df["Caudal"], mode="lines",
              name="Caudal", line=dict(color="#aec7e8", width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_mean.index, y=rolling_mean, mode="lines",
              name="Media móvil", line=dict(color="#d62728", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_std.index, y=rolling_std, mode="lines",
              name="Std móvil", line=dict(color="#ff7f0e", width=2)), row=2, col=1)

fig.update_layout(title="Verificación Visual de Estacionariedad",
                  width=1000, height=550, hovermode="x unified")
fig.update_yaxes(title_text="m³/s", row=1, col=1)
fig.update_yaxes(title_text="m³/s", row=2, col=1)
fig.show()

## 7. Pregunta de Investigación

Con base en los hallazgos del EDA (Semanas 1-3), formulamos la pregunta que guiará el modelado predictivo:

---

### **¿Es posible pronosticar el caudal medio diario de la estación Pueblo Nuevo con un horizonte de 30 días, utilizando modelos de series temporales (SARIMA, Holt-Winters, Prophet) y aprovechando la estacionalidad anual identificada?**

---

**Sub-preguntas:**
1. ¿Qué tan fuerte es la estacionalidad anual y cómo afecta la precisión del pronóstico?
2. ¿Cuál de los tres modelos (SARIMA, Holt-Winters, Prophet) ofrece mejor desempeño predictivo (RMSE, MAE, MAPE)?
3. ¿La transformación logarítmica mejora la capacidad predictiva al estabilizar la varianza?

**Variable objetivo:** Caudal medio diario (m³/s)  
**Horizonte de pronóstico:** 30 días  
**Métrica principal:** RMSE (Root Mean Squared Error)

## 8. Análisis de Estacionalidad Mensual y Trimestral

Profundizamos en los patrones cíclicos del caudal agrupando por **mes** y por **trimestre** para cuantificar la variabilidad estacional.

In [8]:
# Variables temporales
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df["Mes"] = df.index.month
df["Año"] = df.index.year
df["Mes_nombre"] = df["Mes"].map(dict(enumerate(meses, 1)))
df["Trimestre"] = df.index.quarter
df["Trimestre_label"] = "Q" + df["Trimestre"].astype(str)

# Violin plot mensual
fig = px.violin(
    df.reset_index(), x="Mes_nombre", y="Caudal",
    box=True, points=False,
    title="Distribución del Caudal por Mes (Violin Plot)",
    labels={"Mes_nombre": "", "Caudal": "Caudal (m³/s)"},
    color="Mes_nombre",
    color_discrete_sequence=px.colors.qualitative.Set3,
    category_orders={"Mes_nombre": meses},
)
fig.update_layout(width=1000, height=500, showlegend=False)
fig.show()

In [9]:
# Boxplot trimestral
fig = px.box(
    df.reset_index(), x="Trimestre_label", y="Caudal",
    color="Trimestre_label",
    title="Distribución del Caudal por Trimestre",
    labels={"Trimestre_label": "Trimestre", "Caudal": "Caudal (m³/s)"},
    color_discrete_sequence=px.colors.qualitative.Pastel,
    category_orders={"Trimestre_label": ["Q1", "Q2", "Q3", "Q4"]},
)
fig.update_layout(width=700, height=450, showlegend=False)
fig.show()

# Tabla resumen mensual
resumen_mensual = df.groupby("Mes_nombre")["Caudal"].agg(["mean", "std", "median", "min", "max"]).round(2)
resumen_mensual = resumen_mensual.reindex(meses)
resumen_mensual.columns = ["Media", "Desv.Std", "Mediana", "Mínimo", "Máximo"]
print("📊 Resumen estadístico por mes:")
display(resumen_mensual)

📊 Resumen estadístico por mes:


,Media,Desv.Std,Mediana,Mínimo,Máximo
Mes_nombre,,,,,
Ene,3.12,1.51,3.49,0.15,6.44
Feb,2.86,1.48,3.00,0.15,9.39
Mar,3.03,1.58,3.31,0.15,9.39
Abr,3.08,1.44,3.24,0.15,9.39
May,3.25,1.32,3.34,0.62,6.76
Jun,3.70,1.64,3.46,0.67,9.39
Jul,3.92,1.39,3.62,1.22,9.39
Ago,3.79,1.58,3.34,1.43,9.39
Sep,3.50,1.40,3.23,1.22,8.69


## 9. Patrones Interanuales y Tendencia

¿Hay años significativamente más secos o más húmedos? Comparamos el caudal promedio anual y la evolución temporal año a año.

In [10]:
# Caudal promedio anual
anual = df.groupby("Año")["Caudal"].agg(["mean", "std", "median"]).reset_index()
anual.columns = ["Año", "Media", "Desv_Std", "Mediana"]

# Media global
media_global = df["Caudal"].mean()

fig = px.bar(
    anual, x="Año", y="Media",
    error_y="Desv_Std",
    title="Caudal Promedio Anual con Desviación Estándar",
    labels={"Media": "Caudal Promedio (m³/s)", "Año": ""},
    text=anual["Media"].round(1),
    color="Media",
    color_continuous_scale="Blues",
)
fig.add_hline(y=media_global, line_dash="dash", line_color="red",
              annotation_text=f"Media global = {media_global:.1f} m³/s")
fig.update_traces(textposition="outside")
fig.update_layout(width=900, height=500, showlegend=False)
fig.show()

print("📊 Resumen anual:")
display(anual.round(2))

📊 Resumen anual:


,Año,Media,Desv_Std,Mediana
0,2010,3.58,1.79,3.22
1,2011,4.30,0.87,4.09
2,2012,3.25,0.47,3.16
3,2013,3.98,0.61,3.78
4,2014,5.25,1.86,4.78
5,2015,2.91,1.70,2.31
6,2016,1.23,0.86,1.34
7,2017,3.24,0.97,2.92


In [11]:
# Series superpuestas por año para comparar estacionalidad interanual
fig = go.Figure()

colores_anio = px.colors.qualitative.Plotly
for i, (anio, grupo) in enumerate(df.groupby("Año")):
    # Normalizar al día del año para superponer
    grupo_plot = grupo.copy()
    grupo_plot["DiaDelAnio"] = grupo_plot.index.dayofyear
    fig.add_trace(go.Scatter(
        x=grupo_plot["DiaDelAnio"], y=grupo_plot["Caudal"],
        mode="lines", name=str(anio),
        line=dict(width=1.2, color=colores_anio[i % len(colores_anio)]),
    ))

fig.update_layout(
    title="Caudal Diario Superpuesto por Año (Comparación de Estacionalidad)",
    xaxis_title="Día del año", yaxis_title="Caudal (m³/s)",
    width=1000, height=500,
    hovermode="x unified",
    legend_title="Año",
)
fig.show()

## 10. Distribución por Épocas Hidrológicas (Seca vs Lluviosa)

En Colombia, el régimen hidrológico típico se divide en:
- **Época seca:** Dic–Ene–Feb (y Jul–Ago en bimodal)
- **Época lluviosa:** Mar–May y Sep–Nov

Clasificamos los meses en **época seca** y **época lluviosa** para comparar caudales.

In [12]:
# Clasificar meses en épocas hidrológicas
def clasificar_epoca(mes):
    if mes in [12, 1, 2, 7, 8]:
        return "Seca"
    else:
        return "Lluviosa"

df["Epoca"] = df["Mes"].apply(clasificar_epoca)

# Histograma comparativo por época
fig = px.histogram(
    df.reset_index(), x="Caudal", color="Epoca",
    nbins=50, barmode="overlay", opacity=0.7,
    title="Distribución del Caudal por Época Hidrológica",
    labels={"Caudal": "Caudal (m³/s)", "count": "Frecuencia"},
    color_discrete_map={"Seca": "#ff7f0e", "Lluviosa": "#1f77b4"},
)
fig.update_layout(width=900, height=450)
fig.show()

# Estadísticas por época
print("📊 Estadísticas por época hidrológica:")
epoca_stats = df.groupby("Epoca")["Caudal"].describe().round(2)
display(epoca_stats)

📊 Estadísticas por época hidrológica:


,count,mean,std,min,25%,50%,75%,max
Epoca,,,,,,,,
Lluviosa,1704.0,3.38,1.55,0.15,2.60,3.31,3.98,9.39
Seca,1218.0,3.59,1.80,0.15,2.69,3.38,4.17,9.39


In [13]:
# Violin plot por época y año
fig = px.violin(
    df.reset_index(), x="Epoca", y="Caudal",
    color="Epoca", box=True, points=False,
    facet_col="Año", facet_col_wrap=4,
    title="Distribución del Caudal por Época y Año",
    labels={"Caudal": "Caudal (m³/s)", "Epoca": ""},
    color_discrete_map={"Seca": "#ff7f0e", "Lluviosa": "#1f77b4"},
)
fig.update_layout(width=1000, height=600, showlegend=True)
fig.show()

# Proporción de días extremos por época (> percentil 90)
p90 = df["Caudal"].quantile(0.90)
for epoca in ["Seca", "Lluviosa"]:
    subset = df[df["Epoca"] == epoca]
    extremos = (subset["Caudal"] > p90).sum()
    print(f"  {epoca}: {extremos} días con caudal > P90 ({p90:.1f} m³/s) "
          f"→ {extremos/len(subset)*100:.1f}% de sus días")

  Seca: 151 días con caudal > P90 (5.3 m³/s) → 12.4% de sus días
  Lluviosa: 142 días con caudal > P90 (5.3 m³/s) → 8.3% de sus días


## 11. Resumen del EDA: Dashboard de Hallazgos

Consolidamos los resultados clave del EDA en una visualización de resumen.

In [14]:
# Dashboard resumen: 4 gráficos clave
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Serie + Tendencia STL",
        "Componente Estacional (1 ciclo)",
        "ACF (primeros 400 lags)",
        "Caudal Promedio por Mes",
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

# 1. Serie + tendencia
fig.add_trace(go.Scatter(x=df.index, y=df["Caudal"], mode="lines",
              name="Caudal", line=dict(color="#aec7e8", width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=resultado.trend.index, y=resultado.trend, mode="lines",
              name="Tendencia", line=dict(color="#d62728", width=2.5)), row=1, col=1)

# 2. Componente estacional (primeros 365 días)
seasonal_1yr = resultado.seasonal[:365]
fig.add_trace(go.Scatter(x=list(range(365)), y=seasonal_1yr.values, mode="lines",
              name="Estacionalidad", line=dict(color="#2ca02c", width=1.5)), row=1, col=2)

# 3. ACF
acf_400 = acf_vals[:401]
fig.add_trace(go.Bar(x=list(range(401)), y=acf_400,
              marker_color="#1f77b4", name="ACF"), row=2, col=1)
fig.add_hline(y=ic_95, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-ic_95, line_dash="dash", line_color="red", row=2, col=1)

# 4. Promedios mensuales
prom_mes = df.groupby("Mes")["Caudal"].mean()
fig.add_trace(go.Bar(x=meses, y=prom_mes.values,
              marker_color=px.colors.sequential.Blues[3:], name="Promedio mensual"), row=2, col=2)

fig.update_layout(
    title="Dashboard Resumen: EDA Avanzado del Caudal – Estación Pueblo Nuevo",
    width=1100, height=700, showlegend=False,
)
fig.show()

In [15]:
# Resumen numérico del EDA
print("=" * 65)
print("📋 RESUMEN NUMÉRICO DEL EDA AVANZADO")
print("=" * 65)
print(f"🔹 Estación:     Pueblo Nuevo (IDEAM, código 21117100)")
print(f"🔹 Período:      {df.index.min().date()} → {df.index.max().date()} ({len(df)} días)")
print(f"🔹 Variable:     Caudal medio diario (m³/s)")
print(f"\n📊 Estadísticas:")
print(f"   Media:    {df['Caudal'].mean():.2f} m³/s")
print(f"   Mediana:  {df['Caudal'].median():.2f} m³/s")
print(f"   Std:      {df['Caudal'].std():.2f} m³/s")
print(f"   CV:       {df['Caudal'].std()/df['Caudal'].mean()*100:.1f}%")
print(f"\n📐 Descomposición STL:")
ft = 1 - resultado.resid.var() / (resultado.trend + resultado.resid).var()
fs = 1 - resultado.resid.var() / (resultado.seasonal + resultado.resid).var()
print(f"   Fuerza de tendencia:      {ft:.3f}")
print(f"   Fuerza de estacionalidad: {fs:.3f}")
print(f"\n🧪 Estacionariedad:")
print(f"   ADF p-valor (original):     {adf_p_orig:.6f} → {'Estacionaria' if adf_p_orig < 0.05 else 'No estacionaria'}")
print(f"   KPSS p-valor (original):    {kpss_p_orig:.4f} → {'Estacionaria' if kpss_p_orig >= 0.05 else 'No estacionaria'}")
print(f"   ADF p-valor (diferenciada): {adf_p_diff:.6f} → {'Estacionaria' if adf_p_diff < 0.05 else 'No estacionaria'}")
print(f"\n📌 Autocorrelación:")
print(f"   ACF lag 1:   {acf_vals[1]:.4f}")
print(f"   ACF lag 365: {acf_vals[365]:.4f}")
print(f"   PACF lag 1:  {pacf_vals[1]:.4f}")

📋 RESUMEN NUMÉRICO DEL EDA AVANZADO
🔹 Estación:     Pueblo Nuevo (IDEAM, código 21117100)
🔹 Período:      2010-01-01 → 2017-12-31 (2922 días)
🔹 Variable:     Caudal medio diario (m³/s)

📊 Estadísticas:
   Media:    3.47 m³/s
   Mediana:  3.34 m³/s
   Std:      1.66 m³/s
   CV:       48.0%

📐 Descomposición STL:
   Fuerza de tendencia:      0.290
   Fuerza de estacionalidad: -0.032

🧪 Estacionariedad:
   ADF p-valor (original):     0.000104 → Estacionaria
   KPSS p-valor (original):    0.0100 → No estacionaria
   ADF p-valor (diferenciada): 0.000000 → Estacionaria

📌 Autocorrelación:
   ACF lag 1:   0.9343
   ACF lag 365: 0.0395
   PACF lag 1:  0.9346


---

## 12. Conclusiones de la Semana 3

### Hallazgos Clave:

1. **Descomposición STL:** La serie presenta una **estacionalidad anual fuerte** y una tendencia suave. Los residuos muestran variabilidad moderada, lo que indica que la mayor parte de la señal está explicada por tendencia + estacionalidad.

2. **Autocorrelación:** La ACF muestra decaimiento lento y picos significativos en **lag 365** (ciclo anual). La PACF sugiere un componente autorregresivo fuerte en lags cortos (lag 1-3), útil para definir el orden de modelos ARIMA.

3. **Estacionariedad:**
   - Serie original: probable rechazo de estacionariedad por KPSS (presencia de estacionalidad).
   - Serie diferenciada (d=1): se vuelve estacionaria → necesaria **una diferenciación** para modelos ARIMA.

4. **Pregunta de investigación formulada:** Pronóstico de caudal a 30 días usando SARIMA, Holt-Winters y Prophet, evaluados con RMSE, MAE y MAPE.

5. **Estacionalidad mensual:** Se confirma un patrón estacional marcado, con meses de alto caudal y meses de estiaje, coherente con el régimen hidrológico colombiano.

6. **Variación interanual:** Algunos años presentan caudales significativamente diferentes de la media global, lo que podría estar relacionado con fenómenos climáticos (El Niño/La Niña).

7. **Épocas hidrológicas:** La clasificación seca/lluviosa muestra distribuciones claramente diferenciadas, confirmando la utilidad de incluir estacionalidad en los modelos.

### Próximos pasos (Semana 4 – Modelado Predictivo):
- Dividir datos en **train/test** (últimos 30 o 60 días como test)
- Entrenar **SARIMA** con parámetros guiados por ACF/PACF
- Entrenar **Holt-Winters** (Exponential Smoothing triple)
- Entrenar **Prophet** con estacionalidad anual
- Comparar modelos con métricas (RMSE, MAE, MAPE)
- Visualizar pronósticos vs valores reales